# Customer Churn — Data Cleaning Notebook

This notebook cleans `customer_churn.xlsx` following the standard checklist:

1. Check missing values and decide treatment methods
2. Identify and remove duplicate records
3. Correct incorrect data types
4. Fix inconsistent categorical values
5. Identify irrelevant columns
6. Identify possible data leakage
7. Detect and investigate outliers
8. Prepare the cleaned dataset

## 0. Load the data

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

df = pd.read_excel('customer_churn.xlsx')
print(df.shape)
df.head()

(7043, 28)


,Count,Zip Code,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Total Frequency,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV_final,CLTV,Churn Reason
0,1,90003,Male,0,0,0,2,1,0,DSL,1,1,0,0,0,0,Month-to-month,1,0,3,53.85,108.15,Yes,1,86,107.7,107.7,Competitor made better offer
1,1,90005,Female,0,0,1,2,1,0,Fiber optic,0,0,0,0,0,0,Month-to-month,1,0,1,70.70,151.65,Yes,1,67,141.4,141.4,Moved
2,1,90006,Female,0,0,1,8,1,1,Fiber optic,0,0,1,0,1,1,Month-to-month,1,0,5,99.65,820.5,Yes,1,86,797.2,797.2,Moved
3,1,90010,Female,0,1,1,28,1,1,Fiber optic,0,0,1,1,1,1,Month-to-month,1,0,6,104.80,3046.05,Yes,1,84,2934.4,2934.4,Moved
4,1,90015,Male,0,0,1,49,1,1,Fiber optic,0,1,1,0,1,1,Month-to-month,1,1,6,103.70,5036.3,Yes,1,89,5081.3,5081.3,Competitor had better devices


In [2]:
df.dtypes

Count                  int64
Zip Code               int64
Gender                   str
Senior Citizen         int64
Partner                int64
Dependents             int64
Tenure Months          int64
Phone Service          int64
Multiple Lines         int64
Internet Service         str
Online Security        int64
Online Backup          int64
Device Protection      int64
Tech Support           int64
Streaming TV           int64
Streaming Movies       int64
Contract                 str
Paperless Billing      int64
Payment Method         int64
Total Frequency        int64
Monthly Charges      float64
Total Charges         object
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV_final           float64
CLTV                 float64
Churn Reason             str
dtype: object

In [3]:
df.isnull().sum()

Count                0
Zip Code             0
Gender               0
Senior Citizen       0
Partner              0
Dependents           0
Tenure Months        0
Phone Service        0
Multiple Lines       0
Internet Service     0
Online Security      0
Online Backup        0
Device Protection    0
Tech Support         0
Streaming TV         0
Streaming Movies     0
Contract             0
Paperless Billing    0
Payment Method       0
Total Frequency      0
Monthly Charges      0
Total Charges        0
Churn Label          0
Churn Value          0
Churn Score          0
CLTV_final           0
CLTV                 0
Churn Reason         0
dtype: int64

In [4]:
# Total Charges is typed as object -> look for blank/whitespace-only strings hiding as "not missing"
tc_raw = df['Total Charges'].astype(str).str.strip()
blank_mask = tc_raw.eq('')
print("Blank 'Total Charges' entries:", blank_mask.sum())
df.loc[blank_mask, ['Tenure Months', 'Monthly Charges', 'Total Charges']]

Blank 'Total Charges' entries: 11


,Tenure Months,Monthly Charges,Total Charges
2234,0,52.55,
2438,0,20.25,
2568,0,80.85,
2667,0,25.75,
2856,0,56.05,
4331,0,19.85,
4687,0,25.35,
5104,0,20.00,
5719,0,19.70,
6772,0,73.35,


**Finding:** all 11 blanks belong to customers with `Tenure Months = 0` — brand-new customers who
haven't been billed yet. That makes 0 the correct, logical fill value (not the mean/median, which would
invent a charge that was never billed).

**Decision:** convert `Total Charges` to numeric, then fill these 11 rows with 0.

In [5]:
df['Total Charges'] = pd.to_numeric(tc_raw.replace('', np.nan))
df.loc[blank_mask, 'Total Charges'] = 0.0
df['Total Charges'].isnull().sum()  # should be 0 now

np.int64(0)

## 2. Duplicate records

In [6]:
n_dupe = df.duplicated().sum()
print("Fully duplicated rows:", n_dupe)

Fully duplicated rows: 0


**Finding:** 0 exact duplicate rows across all 28 columns.

**Decision:** nothing to remove. Note: the file has no customer ID column, so we can only check for fully
identical rows, not duplicate customers with conflicting attribute values.

## 3. Incorrect data types

In [7]:
df.dtypes

Count                  int64
Zip Code               int64
Gender                   str
Senior Citizen         int64
Partner                int64
Dependents             int64
Tenure Months          int64
Phone Service          int64
Multiple Lines         int64
Internet Service         str
Online Security        int64
Online Backup          int64
Device Protection      int64
Tech Support           int64
Streaming TV           int64
Streaming Movies       int64
Contract                 str
Paperless Billing      int64
Payment Method         int64
Total Frequency        int64
Monthly Charges      float64
Total Charges        float64
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV_final           float64
CLTV                 float64
Churn Reason             str
dtype: object

**Finding:** after Step 1, `Total Charges` is now `float64` (was `object`). Every other column was
already correctly typed:
- Binary indicator columns (`Senior Citizen`, `Partner`, `Dependents`, `Phone Service`, `Multiple Lines`,
  `Online Security`, `Online Backup`, `Device Protection`, `Tech Support`, `Streaming TV`,
  `Streaming Movies`, `Paperless Billing`, `Payment Method`, `Churn Value`) are clean `int64` 0/1 flags.
- `Gender`, `Internet Service`, `Contract`, `Churn Label`, `Churn Reason` are text categories.
- `Tenure Months`, `Churn Score`, `Total Frequency` are `int64`; `Monthly Charges`, `CLTV`, `CLTV_final`
  are `float64`.

**Decision:** no further type conversion needed.

## 4. Inconsistent categorical values

Check every text column for casing/spacing/spelling inconsistencies (e.g. `'Male'` vs `'male '`).

In [8]:
cat_cols = ['Gender', 'Internet Service', 'Contract', 'Churn Label', 'Churn Reason']
for c in cat_cols:
    print(c, '->', sorted(df[c].dropna().unique().tolist()))
    print()

Gender -> ['Female', 'Male']

Internet Service -> ['DSL', 'Fiber optic', 'No']

Contract -> ['Month-to-month', 'One year', 'Two year']

Churn Label -> ['No', 'Yes']

Churn Reason -> ['Attitude of service provider', 'Attitude of support person', 'Competitor had better devices', 'Competitor made better offer', 'Competitor offered higher download speeds', 'Competitor offered more data', 'Deceased', "Don't know", 'Extra data charges', 'Lack of affordable download/upload speed', 'Lack of self-service on Website', 'Limited range of services', 'Long distance charges', 'Moved', 'Network reliability', 'Poor expertise of online support', 'Poor expertise of phone support', 'Price too high', 'Product dissatisfaction', 'Service dissatisfaction']



**Finding:** all categories are already clean and consistently labelled — no stray casing, whitespace,
or spelling variants.

**Decision:** no changes needed.

## 5. Irrelevant columns

In [9]:
print("Count unique values:", df['Count'].unique())
print("CLTV == CLTV_final for all rows:", (df['CLTV'] == df['CLTV_final']).all())
print("Zip Code unique values:", df['Zip Code'].nunique(), "out of", len(df), "rows")

Count unique values: [1]
CLTV == CLTV_final for all rows: True
Zip Code unique values: 1652 out of 7043 rows


**Finding:**
- `Count` is constant (always `1`) — carries no information.
- `CLTV_final` is an exact duplicate of `CLTV` for every row.
- `Zip Code` is a high-cardinality identifier (1,652 unique values) — not directly usable by most models
  without further geo-engineering (e.g. binning into regions), but it isn't meaningless data, so it's a
  judgment call rather than an automatic drop.

**Decision:** drop `Count` and `CLTV_final`. Keep `Zip Code`, flagged as needing extra feature engineering
before modeling.

In [10]:
drop_irrelevant = ['Count', 'CLTV_final']

## 6. Data leakage

Leakage = a column that (directly or indirectly) reveals the target and would not be available at
prediction time in the real world.

In [11]:
print("Corr(Churn Score, Churn Value):", df['Churn Score'].corr(df['Churn Value']))
print()
print("Churn Reason values when Churn Label == 'No':", df.loc[df['Churn Label']=='No', 'Churn Reason'].unique())
print("Churn Reason values when Churn Label == 'Yes' (sample):", df.loc[df['Churn Label']=='Yes', 'Churn Reason'].unique()[:5])

Corr(Churn Score, Churn Value): 0.6648970311816242

Churn Reason values when Churn Label == 'No': <StringArray>
['Don't know']
Length: 1, dtype: str
Churn Reason values when Churn Label == 'Yes' (sample): <StringArray>
['Competitor made better offer', 'Moved', 'Competitor had better devices', 'Competitor offered higher download speeds', 'Competitor offered more data']
Length: 5, dtype: str


**Finding:**
- `Churn Label` is just a text duplicate of the numeric target `Churn Value`.
- `Churn Score` correlates **0.66** with the target — it looks like the output of an existing churn model,
  i.e. a proxy for the answer rather than a predictive input.
- `Churn Reason` is only ever populated with a real reason for customers who churned; every retained
  customer is filled with the placeholder `"Don't know"`. That means the column perfectly encodes the
  target — training on it would make a model that's 100% accurate for the wrong reason and useless in
  production.

**Decision:** exclude `Churn Label`, `Churn Score`, and `Churn Reason` from the model-ready dataset. Keep
`Churn Value` (0/1) as the single target. These three columns are still valuable for *business reporting*
(e.g. "why are people leaving?"), so they're kept in a separate full/reporting dataset — just never fed
into a model alongside the target.

In [12]:
leakage_cols = ['Churn Label', 'Churn Score', 'Churn Reason']

## 7. Outliers

Using the standard 1.5×IQR rule on every numeric column.

In [13]:
num_cols = ['Monthly Charges', 'Total Charges', 'Tenure Months', 'CLTV', 'Churn Score']
outlier_summary = {}
for c in num_cols:
    q1, q3 = df[c].quantile(.25), df[c].quantile(.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[c] < low) | (df[c] > high)).sum())
    outlier_summary[c] = n_out
    print(f"{c:16s} range=({df[c].min()}, {df[c].max()})  IQR bounds=({low:.2f}, {high:.2f})  outliers={n_out}")
outlier_summary

Monthly Charges  range=(18.25, 118.75)  IQR bounds=(-46.02, 171.38)  outliers=0
Total Charges    range=(0.0, 8684.8)  IQR bounds=(-4683.52, 8868.67)  outliers=0
Tenure Months    range=(0, 72)  IQR bounds=(-60.00, 124.00)  outliers=0
CLTV             range=(0.0, 8550.0)  IQR bounds=(-4694.15, 8874.25)  outliers=0
Churn Score      range=(5, 100)  IQR bounds=(-12.50, 127.50)  outliers=0


{'Monthly Charges': 0,
 'Total Charges': 0,
 'Tenure Months': 0,
 'CLTV': 0,
 'Churn Score': 0}

**Finding:** zero IQR outliers in any numeric column — all values fall in plausible ranges (e.g. Tenure
Months 0–72 months, Monthly Charges \$18.25–\$118.75).

**Decision:** no capping, winsorizing, or removal needed.

In [14]:
services = ['Phone Service', 'Multiple Lines', 'Online Security', 'Online Backup', 'Device Protection',
            'Tech Support', 'Streaming TV', 'Streaming Movies']
svc_sum_corr = df['Total Frequency'].corr(df[services].sum(axis=1))
print("Corr(Total Frequency, sum of 8 service flags):", svc_sum_corr)

print()
print("Payment Method value counts:")
print(df['Payment Method'].value_counts())

Corr(Total Frequency, sum of 8 service flags): 0.9999999999999998

Payment Method value counts:
Payment Method
0    3977
1    3066
Name: count, dtype: int64


**`Total Frequency`** correlates perfectly (1.0) with the row-wise sum of the 8 service-flag columns — it's
an exact derived total, not new information. Kept, but flagged: using it *together with* the individual
service flags in a linear model will cause multicollinearity.

**`Payment Method`** only has 2 distinct values (0/1), whereas this field normally has 4 categories
(electronic check, mailed check, bank transfer, credit card) in this type of telecom churn dataset. We
can't recover the original categories from the data alone — left as-is but flagged for the data owner to
confirm what each code means before it's used in analysis.

## 8. Prepare the cleaned dataset

Two outputs:
- **`model_ready`** — irrelevant + leakage columns removed, safe to train a churn model on.
- **`full_clean`** — all columns retained (minus the two truly redundant ones), for business/reporting use.

In [15]:
full_clean = df.drop(columns=['Count', 'CLTV_final']).copy()
model_ready = df.drop(columns=drop_irrelevant + leakage_cols).copy()

print("full_clean :", full_clean.shape)
print("model_ready:", model_ready.shape)
model_ready.head()

full_clean : (7043, 26)
model_ready: (7043, 23)


,Zip Code,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Total Frequency,Monthly Charges,Total Charges,Churn Value,CLTV
0,90003,Male,0,0,0,2,1,0,DSL,1,1,0,0,0,0,Month-to-month,1,0,3,53.85,108.15,1,107.7
1,90005,Female,0,0,1,2,1,0,Fiber optic,0,0,0,0,0,0,Month-to-month,1,0,1,70.70,151.65,1,141.4
2,90006,Female,0,0,1,8,1,1,Fiber optic,0,0,1,0,1,1,Month-to-month,1,0,5,99.65,820.50,1,797.2
3,90010,Female,0,1,1,28,1,1,Fiber optic,0,0,1,1,1,1,Month-to-month,1,0,6,104.80,3046.05,1,2934.4
4,90015,Male,0,0,1,49,1,1,Fiber optic,0,1,1,0,1,1,Month-to-month,1,1,6,103.70,5036.30,1,5081.3


## 9. Document the cleaning decisions

In [16]:
cleaning_log = pd.DataFrame([
    ["Missing values", "11 blank strings in 'Total Charges', all at Tenure Months = 0",
     "Converted to numeric; filled the 11 blanks with 0"],
    ["Duplicates", "0 fully duplicated rows found",
     "No rows removed"],
    ["Data types", "'Total Charges' was object instead of numeric",
     "Converted to float64"],
    ["Inconsistent categories", "Checked Gender, Internet Service, Contract, Churn Label, Churn Reason",
     "All clean — no changes"],
    ["Irrelevant columns", "'Count' constant; 'CLTV_final' 100% duplicate of 'CLTV'; 'Zip Code' high-cardinality",
     "Dropped Count and CLTV_final; kept Zip Code (flagged)"],
    ["Data leakage", "'Churn Label' duplicates target; 'Churn Score' r=0.66 with target; 'Churn Reason' only filled for churned customers",
     "Dropped from model_ready; retained in full_clean for reporting"],
    ["Outliers", "1.5xIQR check on all numeric columns",
     "0 outliers found — no treatment needed"],
    ["Redundant feature (extra)", "'Total Frequency' = exact sum of 8 service flags (r=1.0)",
     "Kept, flagged for multicollinearity risk"],
    ["Data quality flag (extra)", "'Payment Method' has only 2 categories, expected 4",
     "Kept as-is, flagged for data owner to confirm mapping"],
], columns=["Step", "Finding", "Action Taken"])

cleaning_log

,Step,Finding,Action Taken
0,Missing values,"11 blank strings in 'Total Charges', all at Te...",Converted to numeric; filled the 11 blanks with 0
1,Duplicates,0 fully duplicated rows found,No rows removed
2,Data types,'Total Charges' was object instead of numeric,Converted to float64
3,Inconsistent categories,"Checked Gender, Internet Service, Contract, Ch...",All clean — no changes
4,Irrelevant columns,'Count' constant; 'CLTV_final' 100% duplicate ...,Dropped Count and CLTV_final; kept Zip Code (f...
5,Data leakage,'Churn Label' duplicates target; 'Churn Score'...,Dropped from model_ready; retained in full_cle...
6,Outliers,1.5xIQR check on all numeric columns,0 outliers found — no treatment needed
7,Redundant feature (extra),'Total Frequency' = exact sum of 8 service fla...,"Kept, flagged for multicollinearity risk"
8,Data quality flag (extra),"'Payment Method' has only 2 categories, expect...","Kept as-is, flagged for data owner to confirm ..."


## Save outputs

In [17]:
with pd.ExcelWriter('customer_churn_cleaned.xlsx', engine='openpyxl') as writer:
    model_ready.to_excel(writer, sheet_name='Model_Ready_Data', index=False)
    full_clean.to_excel(writer, sheet_name='Full_Cleaned_Data', index=False)
    cleaning_log.to_excel(writer, sheet_name='Cleaning_Log', index=False)

print("Saved customer_churn_cleaned.xlsx")

Saved customer_churn_cleaned.xlsx
